# Benefits Market Intelligence Exploratory Data Analysis: Form 5500 - Schedule A

## Libraries

In [ ]:
# Libraries
import numpy as np
import pandas as pd
import duckdb
import plotly.express as px
from benefits_market_intelligence.config.paths import DB_PATH, FIGURES_PATH, EDA_PATH
from benefits_market_intelligence.visualizations.save import save_figure

FIGURES_PATH.mkdir(parents=True, exist_ok=True)

In [ ]:
# Displaying all
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

## Load Form 5500 - Schedule A data

In [ ]:
# Loading data
with duckdb.connect(DB_PATH, read_only=True) as con:
    SCH_A = con.sql("SELECT * FROM silver.SCH_A").df()

## Viewing data

In [ ]:
# Head of SCH_A
SCH_A.head()

## Shape of data

In [ ]:
# Shape
print(f"Rows: {SCH_A.shape[0]}\nColumns: {SCH_A.shape[1]}")

## Data types

In [ ]:
# Data type counts
SCH_A.dtypes.value_counts()

## Any missing data?

In [ ]:
# Columns with no missing data
SCH_A.isna().sum()[SCH_A.isna().sum() == 0]

In [ ]:
# Missing data
pd.DataFrame(
    {
        "missing_count": SCH_A.isna().sum(),
        "missing_percent": SCH_A.isna().mean().mul(100),
    }
).query("missing_count > 0").sort_values(
    "missing_percent", ascending=False
).rename_axis("column_name").reset_index()

# Average ins_broker_fees_tot_amt by Year

In [ ]:
SCH_A.groupby("FORM_YEAR")["INS_BROKER_FEES_TOT_AMT"].mean().round(3)

## Plotly setup

In [ ]:
# Create paths
FIGURES_PATH.mkdir(parents=True, exist_ok=True)
EDA_PATH.mkdir(parents=True, exist_ok=True)

## Broker compensation over time

In [ ]:
# Average broker commissions and fees by filing year.
cols = [
    c
    for c in ["FORM_YEAR", "INS_BROKER_COMM_TOT_AMT", "INS_BROKER_FEES_TOT_AMT"]
    if c in SCH_A.columns
]
if len(cols) == 3:
    tmp = SCH_A[cols].copy()
    tmp["INS_BROKER_COMM_TOT_AMT"] = pd.to_numeric(
        tmp["INS_BROKER_COMM_TOT_AMT"], errors="coerce"
    )
    tmp["INS_BROKER_FEES_TOT_AMT"] = pd.to_numeric(
        tmp["INS_BROKER_FEES_TOT_AMT"], errors="coerce"
    )
    yearly = tmp.groupby("FORM_YEAR", as_index=False).agg(
        avg_broker_commission=("INS_BROKER_COMM_TOT_AMT", "mean"),
        avg_broker_fees=("INS_BROKER_FEES_TOT_AMT", "mean"),
    )
    long = yearly.melt("FORM_YEAR", var_name="metric", value_name="amount").dropna()
    long["metric"] = long["metric"].map(
        {
            "avg_broker_commission": "Average broker commission",
            "avg_broker_fees": "Average broker fees",
        }
    )
    fig = px.line(
        long,
        x="FORM_YEAR",
        y="amount",
        color="metric",
        markers=True,
        title="Average Broker Compensation Reported on Schedule A Over Time",
        labels={
            "FORM_YEAR": "Filing year",
            "amount": "Average amount ($)",
            "metric": "Compensation type",
        },
    )
    fig.update_yaxes(tickprefix="$", separatethousands=True)
    save_figure(fig, "broker_compensation_over_time", EDA_PATH)
    fig.show()
else:
    print("Skipped: required broker compensation columns were not found.")

## Broker compensation distribution

In [ ]:
# Distribution of reported broker commissions, using a log x-axis because compensation is highly skewed.
if "INS_BROKER_COMM_TOT_AMT" in SCH_A.columns:
    vals = pd.to_numeric(SCH_A["INS_BROKER_COMM_TOT_AMT"], errors="coerce")
    vals = vals[(vals > 0) & np.isfinite(vals)]
    if len(vals):
        fig = px.histogram(
            pd.DataFrame({"broker_commission": vals}),
            x="broker_commission",
            nbins=60,
            log_x=True,
            marginal="box",
            title="Distribution of Reported Broker Commissions",
            labels={
                "broker_commission": "Broker commission ($)",
                "count": "Schedule A records",
            },
        )
        fig.update_xaxes(tickprefix="$", separatethousands=True)
        save_figure(fig, "broker_commission_distribution", EDA_PATH)
        fig.show()
    else:
        print("Skipped: no positive broker commission values.")
else:
    print("Skipped: broker commission column was not found.")

## Broker compensation and covered lives

In [ ]:
## Relationship between broker commissions and people covered by the insurance contract

required = ["INS_BROKER_COMM_TOT_AMT", "INS_PRSN_COVERED_EOY_CNT"]

if all(c in SCH_A.columns for c in required):
    tmp = SCH_A[required].copy()

    for c in required:
        tmp[c] = pd.to_numeric(tmp[c], errors="coerce")

    tmp = tmp[
        (tmp["INS_BROKER_COMM_TOT_AMT"] > 0) & (tmp["INS_PRSN_COVERED_EOY_CNT"] > 0)
    ]

    tmp = tmp[
        np.isfinite(tmp["INS_BROKER_COMM_TOT_AMT"])
        & np.isfinite(tmp["INS_PRSN_COVERED_EOY_CNT"])
    ]

    if len(tmp):
        fig = px.scatter(
            tmp,
            x="INS_PRSN_COVERED_EOY_CNT",
            y="INS_BROKER_COMM_TOT_AMT",
            log_x=True,
            log_y=True,
            opacity=0.55,
            title="Broker Commissions vs. People Covered",
            labels={
                "INS_PRSN_COVERED_EOY_CNT": "People covered at year-end",
                "INS_BROKER_COMM_TOT_AMT": "Broker commission ($)",
            },
        )

        fig.update_yaxes(tickprefix="$", separatethousands=True)

        save_figure(fig, "broker_commission_vs_covered_lives", EDA_PATH)
        fig.show()
    else:
        print("Skipped: no usable positive values.")

else:
    print("Skipped: required covered-lives or commission columns were not found.")

## Insurance carrier concentration

In [ ]:
# Top insurance carriers by number of Schedule A records.
if "INS_CARRIER_NAME" in SCH_A.columns:
    carriers = (
        SCH_A["INS_CARRIER_NAME"]
        .dropna()
        .astype(str)
        .str.strip()
        .replace("", np.nan)
        .dropna()
        .value_counts()
        .head(15)
        .sort_values()
    )
    if len(carriers):
        fig = px.bar(
            carriers,
            x=carriers.values,
            y=carriers.index,
            orientation="h",
            title="Top Insurance Carriers by Schedule A Records",
            labels={"x": "Schedule A records", "y": "Insurance carrier"},
        )
        save_figure(fig, "top_insurance_carriers", EDA_PATH)
        fig.show()
    else:
        print("Skipped: no carrier names available.")
else:
    print("Skipped: insurance carrier name column was not found.")

## Insurance carrier fees

In [ ]:
# Which carriers are associated with the largest total reported broker commissions?
required = ["INS_CARRIER_NAME", "INS_BROKER_COMM_TOT_AMT"]
if all(c in SCH_A.columns for c in required):
    tmp = SCH_A[required].copy()
    tmp["INS_CARRIER_NAME"] = tmp["INS_CARRIER_NAME"].astype(str).str.strip()
    tmp["INS_BROKER_COMM_TOT_AMT"] = pd.to_numeric(
        tmp["INS_BROKER_COMM_TOT_AMT"], errors="coerce"
    )
    tmp = tmp.replace({"INS_CARRIER_NAME": {"": np.nan}})
    carrier_comm = (
        tmp.dropna(subset=["INS_CARRIER_NAME", "INS_BROKER_COMM_TOT_AMT"])
        .groupby("INS_CARRIER_NAME", as_index=False)["INS_BROKER_COMM_TOT_AMT"]
        .sum()
        .sort_values("INS_BROKER_COMM_TOT_AMT", ascending=False)
        .head(15)
        .sort_values("INS_BROKER_COMM_TOT_AMT")
    )
    if len(carrier_comm):
        fig = px.bar(
            carrier_comm,
            x="INS_BROKER_COMM_TOT_AMT",
            y="INS_CARRIER_NAME",
            orientation="h",
            title="Top Insurance Carriers by Reported Broker Commissions",
            labels={
                "INS_BROKER_COMM_TOT_AMT": "Total reported broker commissions ($)",
                "INS_CARRIER_NAME": "Insurance carrier",
            },
        )
        fig.update_xaxes(tickprefix="$", separatethousands=True)
        save_figure(fig, "carrier_broker_commission", EDA_PATH)
        fig.show()
    else:
        print("Skipped: no usable carrier commission data.")
else:
    print("Skipped: required carrier and commission columns were not found.")

## Benefit type mix

In [ ]:
# Count the types of welfare benefits reported on Schedule A.
benefit_cols = {
    "WLFR_BNFT_DENTAL_IND": "Dental",
    "WLFR_BNFT_DRUG_IND": "Prescription drug",
    "WLFR_BNFT_HEALTH_IND": "Health",
    "WLFR_BNFT_HMO_IND": "HMO",
    "WLFR_BNFT_INDEMNITY_IND": "Indemnity",
    "WLFR_BNFT_LIFE_INSUR_IND": "Life insurance",
    "WLFR_BNFT_LONG_TERM_DISAB_IND": "Long-term disability",
    "WLFR_BNFT_PPO_IND": "PPO",
    "WLFR_BNFT_STOP_LOSS_IND": "Stop-loss",
    "WLFR_BNFT_TEMP_DISAB_IND": "Temporary disability",
    "WLFR_BNFT_VISION_IND": "Vision",
}
available = {c: n for c, n in benefit_cols.items() if c in SCH_A.columns}
if available:
    counts = []
    for col, label in available.items():
        v = pd.to_numeric(SCH_A[col], errors="coerce")
        counts.append({"benefit_type": label, "records": int((v == 1).sum())})
    benefits = pd.DataFrame(counts).sort_values("records")
    fig = px.bar(
        benefits,
        x="records",
        y="benefit_type",
        orientation="h",
        title="Reported Welfare Benefit Types on Schedule A",
        labels={
            "records": "Schedule A records reporting benefit",
            "benefit_type": "Benefit type",
        },
    )
    save_figure(fig, "welfare_benefit_mix", EDA_PATH)
    fig.show()
else:
    print("Skipped: no welfare benefit indicator columns were found.")

## Covered lives distribution

In [ ]:
# Distribution of people covered by each insurance contract.
if "INS_PRSN_COVERED_EOY_CNT" in SCH_A.columns:
    vals = pd.to_numeric(SCH_A["INS_PRSN_COVERED_EOY_CNT"], errors="coerce")
    vals = vals[(vals > 0) & np.isfinite(vals)]
    if len(vals):
        fig = px.histogram(
            pd.DataFrame({"covered": vals}),
            x="covered",
            nbins=60,
            log_x=True,
            marginal="box",
            title="Distribution of People Covered by Insurance Contracts",
            labels={
                "covered": "People covered at year-end",
                "count": "Schedule A records",
            },
        )
        save_figure(fig, "covered_lives_distribution", EDA_PATH)
        fig.show()
    else:
        print("Skipped: no positive covered-lives values.")
else:
    print("Skipped: covered-lives column was not found.")

## Broker fee intensity

In [ ]:
# Broker fees per covered person can identify unusually high/low compensation intensity.
required = ["INS_BROKER_FEES_TOT_AMT", "INS_PRSN_COVERED_EOY_CNT"]
if all(c in SCH_A.columns for c in required):
    tmp = SCH_A[required].copy()
    tmp["fees"] = pd.to_numeric(tmp["INS_BROKER_FEES_TOT_AMT"], errors="coerce")
    tmp["covered"] = pd.to_numeric(tmp["INS_PRSN_COVERED_EOY_CNT"], errors="coerce")
    tmp = tmp[(tmp["fees"] > 0) & (tmp["covered"] > 0)]
    tmp["fee_per_person"] = tmp["fees"] / tmp["covered"]
    tmp = tmp[np.isfinite(tmp["fee_per_person"])]
    if len(tmp):
        fig = px.histogram(
            tmp,
            x="fee_per_person",
            nbins=60,
            log_x=True,
            marginal="box",
            title="Distribution of Broker Fees per Covered Person",
            labels={
                "fee_per_person": "Broker fees per covered person ($)",
                "count": "Schedule A records",
            },
        )
        fig.update_xaxes(tickprefix="$", separatethousands=True)
        save_figure(fig, "broker_fee_per_covered_person", EDA_PATH)
        fig.show()
    else:
        print("Skipped: no usable fee-per-person observations.")
else:
    print("Skipped: required broker fee and covered-lives columns were not found.")